# Notebook 2. Feature Engineering and Processed Metrics

Purpose:
- read the base tables from BigQuery
- build daily returns and factor features
- create stock metrics, beta metrics, and correlation metrics
- write the processed layer back into BigQuery


## Imports and Run Settings

This notebook focuses on reusable warehouse features. The calculations stay simple and transparent so the assignment remains easy to defend.


In [1]:
# This cell imports the helpers used to build the processed feature layer.
# The settings stay together so the notebook is easy to rerun after Notebook 1.
from datetime import datetime, timezone

import numpy as np
import pandas as pd

from src.bq_utils import get_bigquery_client, query_to_dataframe, upload_dataframe
from src.config import ANNUALIZATION_DAYS, DATASETS, PROCESSED_TABLES, PROJECT_ID
from src.portfolio_utils import (
    annualize_return,
    annualize_volatility,
    calculate_calmar,
    calculate_sortino,
    classify_stock,
    compute_beta,
    downside_frequency,
    max_drawdown_from_returns,
    sharpe_ratio,
)

RUN_ID = f"run_{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}"
INGESTION_TIMESTAMP = pd.Timestamp.utcnow()
client = get_bigquery_client()


## Load Base Tables

We pull the clean base tables from BigQuery and convert the date columns into pandas datetime values before building returns.


In [2]:
# This cell reads the base stock and factor tables from BigQuery.
# We convert the dates early so the later return calculations stay consistent.
base_stock_prices = query_to_dataframe(
    client,
    f"select * from `{PROJECT_ID}.{DATASETS['processed']}.{PROCESSED_TABLES['base_stock_prices']}`",
)
base_factor_prices = query_to_dataframe(
    client,
    f"select * from `{PROJECT_ID}.{DATASETS['processed']}.{PROCESSED_TABLES['base_factor_prices']}`",
)

base_stock_prices['date'] = pd.to_datetime(base_stock_prices['date'])
base_factor_prices['date'] = pd.to_datetime(base_factor_prices['date'])

print('Base stock rows:', len(base_stock_prices))
print('Base factor rows:', len(base_factor_prices))


ServiceUnavailable: 503 errors resolving bigquerystorage.googleapis.com:443: [field:hostname lookup error:address lookup failed for bigquerystorage.googleapis.com:443: Timeout while contacting DNS servers]

## Returns and Factor Features

This section converts price levels into daily log returns and aligns the factor data so the later stock metrics share the same time index.


In [ ]:
# This cell creates the processed return tables used by the rest of the project.
# We keep the log-return convention consistent across stocks and factors.
stock_base = base_stock_prices.sort_values(['ticker', 'date']).copy()
stock_base['log_return'] = stock_base.groupby('ticker')['close'].transform(
    lambda series: np.log(series / series.shift(1))
)

factor_base = base_factor_prices.sort_values(['factor_name', 'date']).copy()
factor_base['factor_return'] = factor_base.groupby('factor_name')['value'].transform(
    lambda series: np.log(series / series.shift(1))
)
factor_returns = factor_base.dropna(subset=['factor_return']).copy()

factor_wide = factor_returns.pivot_table(
    index='date',
    columns='factor_name',
    values='factor_return',
    aggfunc='mean',
).reset_index()

risk_free_daily = factor_base[factor_base['factor_name'] == 'US10Y'][['date', 'value']].copy()
if risk_free_daily.empty:
    risk_free_daily = pd.DataFrame({'date': factor_base['date'].drop_duplicates().sort_values()})
    risk_free_daily['risk_free_daily'] = 0.0
else:
    risk_free_daily['risk_free_daily'] = risk_free_daily['value'] / 100.0 / ANNUALIZATION_DAYS

asset_returns = stock_base.merge(factor_wide, on='date', how='left').merge(
    risk_free_daily[['date', 'risk_free_daily']],
    on='date',
    how='left',
)
asset_returns['usd_adjusted_return'] = asset_returns['log_return'] - asset_returns['USDARS'].fillna(0)
asset_returns['excess_return'] = asset_returns['log_return'] - asset_returns['risk_free_daily'].fillna(0)
asset_returns['merval_usd_adjusted_return'] = asset_returns['MERVAL'].fillna(0) - asset_returns['USDARS'].fillna(0)
asset_returns['inflation_proxy'] = np.nan
asset_returns['country_risk_proxy'] = np.nan
asset_returns['ingestion_timestamp'] = INGESTION_TIMESTAMP
asset_returns['run_id'] = RUN_ID
asset_returns = asset_returns.dropna(subset=['log_return']).copy()

factor_returns['ingestion_timestamp'] = INGESTION_TIMESTAMP
factor_returns['run_id'] = RUN_ID


## Stock, Beta, and Correlation Metrics

We now create the reusable metrics that support the basket-horizon analysis. This includes return quality, downside-aware metrics, beta, and pairwise correlations.


In [ ]:
# This cell builds the stock-level metrics, beta table, and long correlation matrix.
# The downside metrics help recover the richer assignment scope without adding machine learning.
returns_wide = asset_returns.pivot_table(
    index='date',
    columns='ticker',
    values='log_return',
    aggfunc='mean',
).sort_index()
factor_return_wide = factor_returns.pivot_table(
    index='date',
    columns='factor_name',
    values='factor_return',
    aggfunc='mean',
).sort_index()

beta_rows = []
stock_metric_rows = []

for ticker in returns_wide.columns:
    series = returns_wide[ticker].dropna()
    asset_slice = asset_returns.loc[asset_returns['ticker'] == ticker].sort_values('date').copy()
    asset_slice = asset_slice.set_index('date')
    risk_free_series = asset_slice['risk_free_daily'] if 'risk_free_daily' in asset_slice.columns else 0.0

    average_return = annualize_return(series.mean())
    volatility = annualize_volatility(series.std(ddof=0))
    sharpe = sharpe_ratio(annualize_return(asset_slice['excess_return'].dropna().mean()), volatility)
    sortino = calculate_sortino(asset_slice['log_return'], risk_free_series)
    calmar = calculate_calmar(asset_slice['log_return'])
    max_drawdown = max_drawdown_from_returns(asset_slice['log_return'])
    beta_vs_merval = compute_beta(series, factor_return_wide.get('MERVAL'))
    beta_vs_eem = compute_beta(series, factor_return_wide.get('EEM'))
    corr_with_merval = (
        float(series.corr(factor_return_wide.get('MERVAL')))
        if 'MERVAL' in factor_return_wide.columns
        else np.nan
    )
    corr_with_fx = (
        float(series.corr(factor_return_wide.get('USDARS')))
        if 'USDARS' in factor_return_wide.columns
        else np.nan
    )
    downside_freq = downside_frequency(asset_slice['log_return'])

    stock_profile_row = pd.Series(
        {
            'average_return': average_return,
            'volatility': volatility,
            'beta_vs_merval': beta_vs_merval,
            'sharpe_ratio': sharpe,
        }
    )

    beta_rows.append(
        {
            'ticker': ticker,
            'beta_vs_merval': beta_vs_merval,
            'beta_vs_eem': beta_vs_eem,
            'ingestion_timestamp': INGESTION_TIMESTAMP,
            'run_id': RUN_ID,
        }
    )
    stock_metric_rows.append(
        {
            'ticker': ticker,
            'average_return': average_return,
            'volatility': volatility,
            'beta_vs_merval': beta_vs_merval,
            'beta_vs_eem': beta_vs_eem,
            'corr_with_merval': corr_with_merval,
            'corr_with_fx': corr_with_fx,
            'sharpe_ratio': sharpe,
            'sortino_ratio': sortino,
            'calmar_ratio': calmar,
            'downside_frequency': downside_freq,
            'max_drawdown': max_drawdown,
            'stock_type': classify_stock(stock_profile_row),
            'ingestion_timestamp': INGESTION_TIMESTAMP,
            'run_id': RUN_ID,
        }
    )

beta_metrics = pd.DataFrame(beta_rows)
stock_metrics = pd.DataFrame(stock_metric_rows)
correlation_matrix_long = (
    returns_wide.corr()
    .rename_axis(index='ticker_1', columns='ticker_2')
    .stack()
    .reset_index(name='correlation')
)
correlation_matrix_long['ingestion_timestamp'] = INGESTION_TIMESTAMP
correlation_matrix_long['run_id'] = RUN_ID

display(stock_metrics.head())
display(beta_metrics.head())


,ticker,average_return,volatility,beta_vs_merval,beta_vs_eem,corr_with_merval,corr_with_fx,sharpe_ratio,sortino_ratio,calmar_ratio,downside_frequency,max_drawdown,stock_type,ingestion_timestamp,run_id
0,BMA.BA,0.595033,0.660447,1.222594,0.978273,0.890826,0.036274,0.858213,1.101849,0.968083,0.475843,-0.614650,growth,2026-04-26 02:13:09.691247+00:00,run_20260426_021309
1,BYMA.BA,0.628611,0.520493,0.778283,0.458550,0.719634,0.010411,1.153488,1.405119,1.246054,0.466292,-0.504481,aggressive,2026-04-26 02:13:09.691247+00:00,run_20260426_021309
2,CEPU.BA,0.587054,0.631017,1.122239,0.805652,0.856193,0.033610,0.885595,1.128791,0.941273,0.473596,-0.623681,aggressive,2026-04-26 02:13:09.691247+00:00,run_20260426_021309
3,EDN.BA,0.510349,0.654319,1.045001,0.694096,0.768678,0.017625,0.736828,0.935032,0.669019,0.474157,-0.762832,aggressive,2026-04-26 02:13:09.691247+00:00,run_20260426_021309
4,GGAL.BA,0.580704,0.624192,1.179535,0.937008,0.909364,0.006097,0.885103,1.080302,0.848927,0.473034,-0.684045,aggressive,2026-04-26 02:13:09.691247+00:00,run_20260426_021309


,ticker,beta_vs_merval,beta_vs_eem,ingestion_timestamp,run_id
0,BMA.BA,1.222594,0.978273,2026-04-26 02:13:09.691247+00:00,run_20260426_021309
1,BYMA.BA,0.778283,0.458550,2026-04-26 02:13:09.691247+00:00,run_20260426_021309
2,CEPU.BA,1.122239,0.805652,2026-04-26 02:13:09.691247+00:00,run_20260426_021309
3,EDN.BA,1.045001,0.694096,2026-04-26 02:13:09.691247+00:00,run_20260426_021309
4,GGAL.BA,1.179535,0.937008,2026-04-26 02:13:09.691247+00:00,run_20260426_021309


## Upload Processed Tables

The processed tables written here become the reusable warehouse layer for Notebook 3 and for dbt.


In [ ]:
# This cell writes the processed return and metric tables back to BigQuery.
# Full-refresh mode keeps the feature layer easy to validate from one notebook run to the next.
upload_dataframe(
    client,
    asset_returns,
    PROJECT_ID,
    DATASETS['processed'],
    PROCESSED_TABLES['asset_returns'],
    'WRITE_TRUNCATE',
)
upload_dataframe(
    client,
    factor_returns,
    PROJECT_ID,
    DATASETS['processed'],
    PROCESSED_TABLES['factor_returns'],
    'WRITE_TRUNCATE',
)
upload_dataframe(
    client,
    stock_metrics,
    PROJECT_ID,
    DATASETS['processed'],
    PROCESSED_TABLES['stock_metrics'],
    'WRITE_TRUNCATE',
)
upload_dataframe(
    client,
    beta_metrics,
    PROJECT_ID,
    DATASETS['processed'],
    PROCESSED_TABLES['beta_metrics'],
    'WRITE_TRUNCATE',
)
upload_dataframe(
    client,
    correlation_matrix_long,
    PROJECT_ID,
    DATASETS['processed'],
    PROCESSED_TABLES['correlation_matrix_long'],
    'WRITE_TRUNCATE',
)

print('Notebook 2 tables written:')
for table_name in [
    'asset_returns',
    'factor_returns',
    'stock_metrics',
    'beta_metrics',
    'correlation_matrix_long',
]:
    print(f"- {PROJECT_ID}.{DATASETS['processed']}.{PROCESSED_TABLES[table_name]}")


Notebook 2 tables written:
- bigdata-financeargentina.processed_market.asset_returns
- bigdata-financeargentina.processed_market.factor_returns
- bigdata-financeargentina.processed_market.stock_metrics
- bigdata-financeargentina.processed_market.beta_metrics
- bigdata-financeargentina.processed_market.correlation_matrix_long
